# Goodreads Book Genre Trends — SQL Analysis

## Overview

This notebook uses SQL queries against the SQLite database created during the database-creation stage.

The purpose of this analysis is to demonstrate how a relational database can be used to investigate trends in Goodreads book publication, ratings, and genre classifications.

The analysis focuses on relationships between books, authors, genres, and publication years.

The SQL queries in this notebook include intermediate and advanced SQL techniques such as:

- JOINs
- GROUP BY
- Aggregate functions
- HAVING
- Subqueries
- Common Table Expressions (CTEs)
- Window functions

These techniques satisfy the capstone requirement to include at least three intermediate/advanced SQL queries.

## 2. Connect to the SQLite Database

The SQLite database created during the database-creation stage is opened using Python's `sqlite3` library.

SQL queries will be executed against this database, and the results will be loaded into Pandas DataFrames for inspection and further analysis.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("../Data/goodreads_capstone.db")

conn = sqlite3.connect(db_path)

# Make sure SQLite enforces foreign-key relationships
conn.execute("PRAGMA foreign_keys = ON;")

print("Connected to:", db_path)

Connected to: ..\Data\goodreads_capstone.db


## 3. Query 1 — Number of Books by Genre

This query determines how many unique books are associated with each Goodreads genre.

The query demonstrates:

- `JOIN`
- `GROUP BY`
- `COUNT()`
- `ORDER BY`

The `books`, `book_genres`, and `genres` tables are joined through their primary and foreign keys.

Because a book can belong to multiple genres, the `book_genres` relationship table is necessary to connect books to their genre classifications.

In [2]:
query1 = """
SELECT
    g.genre_name,
    COUNT(DISTINCT bg.book_id) AS book_count
FROM genres AS g
JOIN book_genres AS bg
    ON g.genre_id = bg.genre_id
GROUP BY g.genre_id, g.genre_name
ORDER BY book_count DESC;
"""

genre_counts = pd.read_sql_query(query1, conn)

display(genre_counts.head(20))

,genre_name,book_count
0,romance,199105
1,history,173414
2,fantasy,143890
3,childrens,116001
4,contemporary,94620
5,comics,91556
6,mystery,89657
7,audiobook,76260
8,science fiction,75022
9,young adult,71370


## 4. Query 2 — Genres With at Least 1,000 Books

This query identifies genres with a substantial number of books in the database.

The query demonstrates:

- `JOIN`
- `GROUP BY`
- `COUNT()`
- `HAVING`
- `ORDER BY`

Unlike a `WHERE` clause, which filters individual records before aggregation, `HAVING` filters groups after the aggregation has been performed.

Only genres associated with at least 1,000 unique books are returned.

In [3]:
query2 = """
SELECT
    g.genre_name,
    COUNT(DISTINCT bg.book_id) AS book_count
FROM genres AS g
JOIN book_genres AS bg
    ON g.genre_id = bg.genre_id
GROUP BY g.genre_id, g.genre_name
HAVING COUNT(DISTINCT bg.book_id) >= 1000
ORDER BY book_count DESC;
"""

large_genres = pd.read_sql_query(query2, conn)

display(large_genres)

,genre_name,book_count
0,romance,199105
1,history,173414
2,fantasy,143890
3,childrens,116001
4,contemporary,94620
...,...,...
436,magick,1010
437,graphic novels manga,1008
438,martial arts,1006
439,weird fiction,1005


## 5. Query 3 — Books Rated Above the Overall Average

This query identifies books whose Goodreads star rating is higher than the average rating of all books in the database.

The query uses a subquery to calculate the overall average rating.

The outer query then compares each book's rating against that calculated value.

This demonstrates the use of a subquery as an intermediate/advanced SQL technique.

In [4]:
query3 = """
SELECT
    book_id,
    name,
    pub_year,
    star_rating,
    num_ratings
FROM books
WHERE star_rating > (
    SELECT AVG(star_rating)
    FROM books
)
ORDER BY star_rating DESC, num_ratings DESC;
"""

above_average_books = pd.read_sql_query(query3, conn)

display(above_average_books.head(20))

,book_id,name,pub_year,star_rating,num_ratings
0,1101336,Paul Simon's Concert in the Park,1991,5.0,41
1,960271,"The Evil That Men Do: Faith, Injustice and the...",2016,5.0,32
2,1146334,The Wills Eye Manual: Office and Emergency Roo...,2016,5.0,31
3,377940,Vow of Blood,2008,5.0,27
4,137758,Bay Poetics,2006,5.0,26
5,1393574,The Selfie,2016,5.0,22
6,1455068,Let Today Be A Holiday : 365 Ways to Co-Create...,2006,5.0,22
7,1041274,Steel Phantom,2016,5.0,20
8,1098435,Rhapsody in Blue,1976,5.0,20
9,179420,Photorealism,1980,5.0,19


## 6. Query 4 — Average Rating by Genre

This query examines the average Goodreads rating associated with each genre.

Because books may belong to multiple genres, the query joins the `books`, `book_genres`, and `genres` tables.

The query also calculates the number of books represented in each genre.

Genres with fewer than 100 books are excluded to reduce the influence of very small samples.

This query uses:

- Multiple `JOIN`s
- `GROUP BY`
- Aggregate functions
- `HAVING`
- `ORDER BY`

In [5]:
query4 = """
SELECT
    g.genre_name,
    COUNT(DISTINCT b.book_id) AS book_count,
    ROUND(AVG(b.star_rating), 2) AS average_rating
FROM books AS b
JOIN book_genres AS bg
    ON b.book_id = bg.book_id
JOIN genres AS g
    ON bg.genre_id = g.genre_id
GROUP BY g.genre_id, g.genre_name
HAVING COUNT(DISTINCT b.book_id) >= 100
ORDER BY average_rating DESC;
"""

genre_ratings = pd.read_sql_query(query4, conn)

display(genre_ratings.head(20))

,genre_name,book_count,average_rating
0,baha i,101,4.53
1,devotional,1527,4.33
2,lds non fiction,270,4.33
3,colouring books,188,4.31
4,prayer,1444,4.29
5,african american romance,964,4.28
6,field guides,558,4.28
7,scripture,325,4.28
8,ornithology,139,4.27
9,catholic,4079,4.24


## 7. Query 5 — Books Published by Year

This query examines the number of unique books published during each year represented in the database.

The results can be used to identify changes in book publication volume over time.

This query uses:

- `GROUP BY`
- `COUNT()`
- `ORDER BY`

The results will later be useful for visualizing publication trends.

In [6]:
query5 = """
SELECT
    pub_year,
    COUNT(*) AS book_count
FROM books
GROUP BY pub_year
ORDER BY pub_year;
"""

publication_trends = pd.read_sql_query(query5, conn)

display(publication_trends.head(20))

,pub_year,book_count
0,1900,711
1,1901,383
2,1902,385
3,1903,370
4,1904,410
5,1905,444
6,1906,403
7,1907,431
8,1908,439
9,1909,457


## 8. Query 6 — Genre Trends Over Time

This query combines publication year and genre information to determine how many books were associated with each genre during each publication year.

The query joins:

- `books`
- `book_genres`
- `genres`

The results are grouped by publication year and genre.

This creates a dataset that can be used to investigate changes in genre popularity and representation over time.

The query demonstrates multiple table joins, grouping, aggregation, and ordering.

In [7]:
query6 = """
SELECT
    b.pub_year,
    g.genre_name,
    COUNT(DISTINCT b.book_id) AS book_count
FROM books AS b
JOIN book_genres AS bg
    ON b.book_id = bg.book_id
JOIN genres AS g
    ON bg.genre_id = g.genre_id
GROUP BY
    b.pub_year,
    g.genre_id,
    g.genre_name
ORDER BY
    b.pub_year,
    book_count DESC;
"""

genre_trends = pd.read_sql_query(query6, conn)

display(genre_trends.head(20))

,pub_year,genre_name,book_count
0,1900,classics,142
1,1900,history,109
2,1900,childrens,82
3,1900,picture books,56
4,1900,short stories,55
5,1900,reference,53
6,1900,poetry,50
7,1900,literature,46
8,1900,historical fiction,42
9,1900,philosophy,40


## 9. Query 7 — High-Rated Genres Using a CTE

This query uses a Common Table Expression (CTE) to first calculate the number of books and average rating for each genre.

The outer query then filters the aggregated results to identify genres that have both:

- At least 1,000 books
- An average rating of at least 4.0

Using a CTE makes the query easier to read by separating the aggregation step from the filtering step.

This represents an additional advanced SQL technique beyond the three required intermediate/advanced queries.

In [8]:
query7 = """
WITH genre_statistics AS (
    SELECT
        g.genre_id,
        g.genre_name,
        COUNT(DISTINCT b.book_id) AS book_count,
        AVG(b.star_rating) AS average_rating
    FROM genres AS g
    JOIN book_genres AS bg
        ON g.genre_id = bg.genre_id
    JOIN books AS b
        ON bg.book_id = b.book_id
    GROUP BY
        g.genre_id,
        g.genre_name
)

SELECT
    genre_name,
    book_count,
    ROUND(average_rating, 2) AS average_rating
FROM genre_statistics
WHERE book_count >= 1000
  AND average_rating >= 4.0
ORDER BY average_rating DESC;
"""

high_rated_genres = pd.read_sql_query(query7, conn)

display(high_rated_genres)

,genre_name,book_count,average_rating
0,devotional,1527,4.33
1,prayer,1444,4.29
2,catholic,4079,4.24
3,comic strips,1163,4.23
4,christian non fiction,3161,4.21
...,...,...,...
62,psychoanalysis,1199,4.01
63,western romance,2021,4.01
64,birds,2253,4.00
65,military science fiction,1222,4.00


## 10. SQL Analysis Summary

The SQL analysis demonstrates how the relational database can be used to investigate the Goodreads dataset from multiple perspectives.

The queries demonstrated several intermediate and advanced SQL techniques, including:

| Query | SQL Techniques | Purpose |
|---|---|---|
| Query 1 | JOIN, GROUP BY, COUNT | Books by genre |
| Query 2 | JOIN, GROUP BY, HAVING | Genres with substantial book counts |
| Query 3 | Subquery, AVG | Books rated above the overall average |
| Query 4 | JOIN, GROUP BY, HAVING, AVG | Average rating by genre |
| Query 5 | GROUP BY, COUNT | Publication volume by year |
| Query 6 | Multiple JOINs, GROUP BY, COUNT | Genre trends over time |
| Query 7 | CTE, JOIN, GROUP BY, HAVING/WHERE | High-rated genres with sufficient sample sizes |

The results from these queries provide the foundation for the exploratory data analysis and visualization stages of the project.

In particular, the publication-year and genre-trend queries can be used to investigate how the representation of different genres has changed over time.

Fantasy was among the genres with the largest number of associated books, while several smaller genres had substantially fewer records. This demonstrates the importance of applying minimum sample-size thresholds when comparing genre-level averages.

## 11. Close Database Connection

The SQLite database connection is closed after all SQL analysis has been completed.

The resulting query datasets can be used in the subsequent visualization and exploratory analysis stages of the capstone project.

In [9]:
conn.close()

print("Database connection closed.")

Database connection closed.
